## Tujuan Pembelajaran

* Membuka, membaca, dan menulis file teks menggunakan open()
* Memahami perbedaan mode file (r, w, a) dan dampak berbahaya dari mode w kalau tidak hati-hati
* Membaca file dengan berbagai cara (.read(), .readline(), .readlines(), iterasi langsung) dan tahu kapan pakai yang mana
* Menggunakan context manager (with) sebagai cara standar dan aman membuka file
* Membersihkan teks hasil baca file (newline, whitespace) menggunakan .strip() dari materi 02

## Isi Materi

* File I/O — konsep file, path, open()
* File Modes — r, w, a, dan bahaya mode w
* Reading — empat cara membaca file
* Writing — .write() dan .writelines()
* Context Manager — with, penutupan otomatis, alasannya
* Text Processing — newline, .strip(), kesadaran encoding

## Goals
* Menjelaskan kenapa file harus ditutup setelah selesai dipakai
*  Menjelaskan perbedaan mode r, w, a — terutama kenapa w bisa menghapus data tanpa peringatan
* Memilih cara membaca file yang tepat sesuai kebutuhan (.read() vs .readlines() vs iterasi langsung)
*  Menulis pola with open(...) as f: sebagai kebiasaan default, bukan open() manual
* Membersihkan karakter \n dari hasil baca file menggunakan .strip()

# 1. File I/O

## Konsep File

* File adalah data yang tersimpan secara permanen di penyimpanan (disk), bukan cuma di memori sementara seperti variabel
* Bedanya dengan variabel biasa: variabel hilang begitu program berhenti, file tetap ada sampai dihapus manual
* Ini kenapa file penting untuk data science — dataset yang kamu olah biasanya berasal dari file (CSV, JSON, txt), dan hasil olahanmu biasanya perlu disimpan kembali ke file

## File path

* Path : alamat atau lokasi file, ada dua jenis path
    * relative path: lokasi file relatif terhadap tempat kode dijalankan
        - (contoh: "data.txt" atau "folder/data.txt")
    * Absolute path: path lengkap dari lokasi root sistem
        - (contoh: "D:/DS_Road/data-science-learning/data.txt")
* Untuk belajar, relative path biasanya cukup — asal file-nya ada di folder yang sama dengan notebook

In [1]:
# Relative path -> file harus ada di folder yang sama dengan notebook ini
path_relatif = "catatan.txt"

# Absolute path -> lokasi lengkap, tidak tergantung dari mana kode dijalankan
path_absolut = "D:/DS_Road/data-science-learning/catatan.txt"

## `open()`

* function python untuk membuka koneksi ke sebuah file, sebelum bisa dibaca / ditulis
* Argumen pertama: path file. Argumen kedua: mode

In [ ]:
# Membuka file untuk ditulis 
file = open("catatan.txt", "w")
file.write("File pertama")
file.close()  # WAJIB ditutup manual kalau pakai cara ini -> nanti akan digantikan 'with'

> Catatan penting: open() yang ditulis manual seperti di atas wajib diikuti .close(). Kalau lupa, file bisa "menggantung" (masih dianggap sedang dipakai oleh program), yang bisa menyebabkan data tidak tersimpan sepenuhnya atau file terkunci dari program lain. Nanti di bagian 5, kamu akan belajar cara yang jauh lebih aman supaya tidak perlu khawatir lupa .close().

# 2. File Modes

## `r` read

* Mode default jika mode tidak ditulis, hanya untuk membaca
* Kalau file yang mau dibuka tidak ada, akan error (FileNotFoundError)

In [ ]:
file = open("catatan.txt", "r")
isi = file.read()
print(isi)
file.close()

File pertama


## `w` write

* Untuk menulis file
* Kalau file belum ada, otomatis dibuat baru
* Kalau file sudah ada — ini bagian krusial di bawah

In [13]:
catat = open("catatan.txt", "w")
catat.write("Baris 2 22222222222222")
catat.close()

print(open("catatan.txt", "r"))

<_io.TextIOWrapper name='catatan.txt' mode='r' encoding='cp1252'>


## `a` (append)

* Untuk menambahkan tulisan ke akhir file yang sudah ada, tanpa menghapus isi sebelumnya
* Kalau file belum ada, otomatis dibuat baru (sama seperti w)

## Dampak mode w

* Ini yang paling penting dipahami di seluruh bagian ini. Membuka file dengan mode w akan langsung mengosongkan seluruh isi file lama, bahkan sebelum kamu menulis apa pun — proses ini terjadi seketika saat open() dipanggil, bukan saat .write() dipanggil
* Ini beda jauh dari mode a yang aman menambahkan tanpa menghapus

In [4]:
# Simulasikan file yang sudah berisi data penting
with open("data_penting.txt", "w") as f:
    f.write("Data penting jangan sampai hilang!")

# Sekarang bayangkan kamu TIDAK SENGAJA membuka lagi dengan mode 'w'
with open("data_penting.txt", "w") as f:
    pass   # bahkan TANPA menulis apa pun, isi lama sudah hilang!

with open("data_penting.txt", "r") as f:
    print(repr(f.read()))   # '' -> KOSONG! Data lama sudah terhapus permanen

# Bandingkan dengan mode 'a' -> aman, tidak menghapus
with open("data_penting.txt", "a") as f:
    f.write("Baris baru ditambahkan")

with open("data_penting.txt", "r") as f:
    print(f.read())   # "Baris baru ditambahkan" -> aman ditambahkan, bukan menimpa

''
Baris baru ditambahkan


> Kesalahan umum paling merugikan di seluruh materi ini: membuka file yang berisi data penting dengan mode w untuk sekadar "melihat isinya", padahal maksudnya mau r. Begitu open(path, "w") dieksekusi, isi lama langsung hilang tanpa peringatan apa pun dan tidak bisa dikembalikan. Selalu double-check mode sebelum menjalankan kode yang menyentuh file berisi data penting.

# 3. Reading

## `.read()`

* Membaca seluruh isi file, dikembalian sbg satu string besar
* Cocok untuk file kecil, kurang cocok untuk file sangat besar (semua harus masuk memori sekaligus)

In [5]:
with open("catatan.txt", "r") as f:
    isi = f.read()
    
print(isi)
print(type(isi))

File pertama
<class 'str'>


## `readline()`

* Membaca satu baris saja, setiap kali dipanggil akan lanjut ke baris berikutnya
* Berguna kalau kamu cuma butuh beberapa baris pertama, tanpa membaca semuanya

In [6]:
with open("catatan.txt", "r") as f:
    baris1 = f.readline()
    baris2 = f.readline()
print(baris1)   # baris pertama file (termasuk karakter \n di akhirnya)
print(baris2)   # baris kedua file

File pertama



## `.readlines()`

* Membaca seluruh baris sekaligus, dikembalikan sebagai list of strings — satu elemen list per baris
* Mirip .read() tapi hasilnya sudah "dipecah" per baris, mirip .split("\n") (materi 02) tapi bawaan Python

In [14]:
with open("catatan.txt", "r") as f:
    semua_baris = f.readlines()
print(semua_baris)   # ['baris 1\n', 'baris 2\n', 'baris 3']
print(type(semua_baris))   # <class 'list'>
print(len(semua_baris))    # jumlah baris dalam file

['Baris 2 22222222222222']
<class 'list'>
1


## Iterasi file

File object langsung bisa di-iterasi dengan for (materi 08), satu baris per putaran — ini cara paling hemat memori untuk file besar, karena baris dibaca satu-satu, bukan semuanya sekaligus masuk memori seperti .readlines()

In [15]:
with open("catatan.txt", "r") as f:
    for baris in f:
        print(baris.strip())

Baris 2 22222222222222


## Ringkasan kapan pakai yang mana:

* .read() → butuh seluruh isi sebagai satu string utuh (misal untuk dicari kata tertentu di seluruh teks)
* .readlines() → butuh seluruh baris sebagai list, dan file-nya tidak terlalu besar
* Iterasi langsung (for baris in f) → memproses baris demi baris, terutama untuk file besar

# 4. Writing

## `write()`

* Menulis satu string ke file
* Baris baru harus menambahkan \n

In [ ]:
with open("output.txt", "w") as f:
    f.write("Baris pertama")
    f.write("Baris kedua")
    
with open("output.txt", "r") as f:
    print(f.read())
    
# "Baris pertamaBaris kedua" -> nyambung, bukan 2 baris!

Baris pertamaBaris kedua


Versi benar

In [17]:
# Versi yang benar -> tambahkan \n secara eksplisit
with open("output.txt", "w") as f:
    f.write("Baris pertama\n")
    f.write("Baris kedua\n")

with open("output.txt", "r") as f:
    print(f.read())
# Baris pertama
# Baris kedua

Baris pertama
Baris kedua



## .writelines()

* Menulis list of strings sekaligus ke file
* Nama method-nya agak menjebak: .writelines() juga tidak menambahkan \n otomatis antar elemen — kamu tetap harus menyertakan \n di tiap string dalam list-nya

In [18]:
daftar_baris = ["Item 1\n", "Item 2\n", "Item 3\n"]   # \n WAJIB disertakan manual

with open("daftar.txt", "w") as f:
    f.writelines(daftar_baris)

with open("daftar.txt", "r") as f:
    print(f.read())
# Item 1
# Item 2
# Item 3

# Kalau lupa \n -> hasilnya nyambung semua, sama seperti kasus .write() di atas
daftar_salah = ["Item 1", "Item 2", "Item 3"]
with open("daftar_salah.txt", "w") as f:
    f.writelines(daftar_salah)

with open("daftar_salah.txt", "r") as f:
    print(f.read())   # "Item 1Item 2Item 3" -> nyambung jadi satu baris

Item 1
Item 2
Item 3

Item 1Item 2Item 3


# 5. Context Manager


## Konsep with

* with adalah cara Python menangani sumber daya (seperti file) yang perlu dibuka lalu wajib ditutup — disebut context manager
* Alih-alih menulis open() dan .close() terpisah (rawan lupa), with menggabungkan keduanya jadi satu blok yang otomatis rapi

## `with open()`

In [19]:
# Pola standar yang akan kamu pakai TERUS MENERUS mulai sekarang
with open("catatan.txt", "r") as f:
    isi = f.read()
    print(isi)
# di luar blok 'with' ini, file SUDAH otomatis tertutup, tanpa perlu f.close()

print(f.closed)   # True -> terbukti sudah tertutup otomatis

Baris 2 22222222222222
True


## Automatic closing

* File akan otomatis ditutup begitu blok with selesai — bahkan kalau terjadi error di tengah proses membaca/menulis
* Ini yang tidak dijamin kalau kamu pakai open() + .close() manual: kalau error terjadi sebelum baris .close() dieksekusi, file tidak akan pernah tertutup

In [20]:
# Cara manual -> BERISIKO kalau ada error sebelum .close()
file = open("catatan.txt", "r")
isi = file.read()
angka = 10 / 0          # error terjadi di sini!
file.close()              # baris ini TIDAK PERNAH dijalankan karena error di atas

# Cara 'with' -> AMAN, file tetap tertutup walau ada error
with open("catatan.txt", "r") as f:
    isi = f.read()
    # angka = 10 / 0    -> seandainya error terjadi di sini,
    #                       file TETAP otomatis tertutup oleh 'with'

ZeroDivisionError: division by zero

## Mengapa context manager digunakan

* Mencegah resource leak — file yang tidak tertutup bisa menghabiskan sumber daya sistem kalau terjadi berkali-kali dalam program besar
* Kode lebih ringkas — tidak perlu menulis .close() terpisah, dan tidak perlu mengingat-ingat "sudah ditutup belum ya"
* Aman dari error — seperti dibuktikan di atas, with menjamin penutupan bahkan saat terjadi exception di tengah jalan
* Karena alasan-alasan ini, with open(...) adalah cara standar yang dipakai hampir seluruh kode Python profesional — pola open() + .close() manual sebaiknya kamu anggap sebagai "cara lama" yang jarang dipakai lagi

# 6. Text Processing

## Newline \n

* Saat file dibaca per baris (.readlines() atau iterasi langsung), setiap baris tetap membawa karakter \n di akhirnya (kecuali kadang baris terakhir file)
* \n adalah karakter khusus yang berarti "pindah baris" — tidak terlihat sebagai tulisan, tapi tetap bagian dari string

In [22]:
with open("catatan.txt", "r") as f:
    baris = f.readline()
print(baris)
print(repr(baris)) # 'baris pertama\n' -> repr() membongkar karakter tersembunyi

Baris 2 22222222222222
'Baris 2 22222222222222'


## .strip()

* Solusi paling umum untuk membuang \n (dan whitespace lain) di awal/akhir string — sudah kamu pelajari di materi 02, sekarang kamu lihat kegunaan nyatanya
* Sangat penting dipakai setiap kali memproses baris hasil baca file, supaya tidak ada karakter tersembunyi yang mengganggu perbandingan/pengolahan data

In [31]:
with open("catatan.txt", "w") as f:
    f.write("Halo ")

In [32]:
with open("catatan.txt", "r") as f:
    for baris in f:
        baris_bersih = baris.strip()   # buang \n dan spasi di pinggir
        print(repr(baris_bersih))       # sekarang tidak ada \n lagi

# Contoh kenapa ini penting: perbandingan string bisa gagal kalau lupa strip()
with open("catatan.txt", "r") as f:
    baris_pertama = f.readline()

if baris_pertama == "Halo":          # ini kemungkinan besar FALSE!
    print("Cocok")
else:
    print("Tidak cocok")   # ini yang tercetak, karena baris_pertama sebenarnya "Halo\n"

# Perbaikan
if baris_pertama.strip() == "Halo":
    print("Cocok sekarang")   # ini yang tercetak setelah di-strip()

'Halo'
Tidak cocok
Cocok sekarang


## Basic encoding awareness

* Encoding adalah aturan bagaimana karakter (huruf, simbol) diterjemahkan jadi angka biner yang disimpan komputer
* Python secara default (di kebanyakan sistem) memakai encoding UTF-8 — cukup untuk hampir semua kasus modern, termasuk karakter non-Latin
* Kadang kamu akan menemukan file lama/dari sumber tertentu yang encoding-nya berbeda (misal latin-1, cp1252), yang bisa menyebabkan error UnicodeDecodeError saat dibuka dengan asumsi UTF-8
* Solusi dasarnya: tentukan encoding secara eksplisit lewat parameter encoding

In [ ]:
# Kalau menemukan error UnicodeDecodeError, coba tentukan encoding eksplisit
with open("file_lama.txt", "r", encoding="utf-8") as f:
    isi = f.read() 

# Kalau UTF-8 masih error, kadang encoding aslinya beda, misal:
with open("file_lama.txt", "r", encoding="latin-1") as f:
    isi = f.read()

1. Membaca .txt

Buat file belanja.txt menggunakan with open(..., "w"), isinya (gunakan \n di tiap akhir baris):
```text
  Beras 50000
  Telur 28000
  Minyak 32000
  Gula 15000
```
Buka kembali file itu dengan mode "r", baca seluruh isinya dengan .read(), lalu cetak

In [35]:
with open("belanja.txt", "w") as f:
    f.write(" Beras 50000\n Telur 28000\n Minyak 32000\n Gula 15000")

with open("belanja.txt", "r") as f:
    print(f.read())
    

 Beras 50000
 Telur 28000
 Minyak 32000
 Gula 15000


2. Menghitung jumlah baris

    * Dari file belanja.txt yang sama, gunakan .readlines() untuk membaca semua baris ke sebuah list
    * Cetak jumlah baris menggunakan len()
    * Gunakan iterasi langsung (for baris in f) untuk mencetak tiap baris dengan format f"Baris {nomor}: {isi}" (pakai enumerate() dari materi 08, mulai dari 1)

In [40]:
with open("belanja.txt", "r") as f:
    for nomor, baris in enumerate(f):
        print(f"Baris {nomor}: {baris}")
    print(len(baris))

Baris 0:  Beras 50000

Baris 1:  Telur 28000

Baris 2:  Minyak 32000

Baris 3:  Gula 15000
11


3. Membersihkan text

    * Dari file belanja.txt, baca tiap baris, lalu:
    * Gunakan .strip() untuk membuang \n
    * Gunakan .split() (materi 02) untuk memisahkan nama barang dan harga (pisahkan berdasarkan spasi)
    * Simpan hasilnya ke dalam list of dictionaries (materi 05), format: {"nama": "Beras", "harga": 50000} — ingat, hasil .split() semuanya string, jadi harga perlu diubah ke int() (materi 02)
    * Cetak list of dictionaries hasilnya

In [52]:
belanja = []

with open("belanja.txt", "r") as f:
    for baris in f:
        baris = baris.strip()
        nama, harga = baris.split()
        harga = int(harga)
        
        item = {
            "Nama": nama,
            "Harga": harga
        }
        belanja.append(item)

print(belanja)

[{'Nama': 'Beras', 'Harga': 50000}, {'Nama': 'Telur', 'Harga': 28000}, {'Nama': 'Minyak', 'Harga': 32000}, {'Nama': 'Gula', 'Harga': 15000}]


4. Menulis hasil ke file baru

    * Dari list of dictionaries hasil latihan 3, hitung total belanja (pakai for loop + akumulator, dari materi 08/09)
    * Buat function format_laporan(data_belanja, total) yang me-return satu string laporan lengkap (gunakan f-string dan \n untuk tiap baris), format bebas tapi minimal berisi rincian tiap barang dan total di akhir
    * Tulis hasil string itu ke file baru bernama laporan_belanja.txt menggunakan with open(..., "w")
    * Buka kembali laporan_belanja.txt dengan mode "r" untuk membuktikan isinya benar sudah tersimpan